# 03 — Weather Feature Engineering

**Phase 2, Step 2** of the forecasting pipeline.

**Input:**   `data/interim/weather_daily_clean.csv`  (from notebook 02)
**Output:**  `data/processed/weather_features.csv`

---

## What we build and why

| Family | What it captures | Why it helps forecasting |
|---|---|---|
| **Wind vectors** | `wind_u`, `wind_v`, `sin/cos` of direction | Makes circular wind direction usable by linear/tree models |
| **Lag features** | `col_lag_k` for k ∈ {1,2,3,7,14,30,365} | Autoregressive memory — yesterday, last week, last year |
| **Rolling stats** | `col_roll_w_agg` for w ∈ {3,7,14,30}, agg ∈ {mean,std,min,max} | Smooths noise; captures recent trend and volatility |
| **Differences** | `col_diff_k` for k ∈ {1,7} | Stationarity — SARIMA-style differencing as explicit features |
| **Calendar** | `day_of_year`, `month`, `quarter`, `is_weekend`, … | Captures human-scale periodicity |
| **Fourier** | `annual_sin_k`, `annual_cos_k` for k ∈ {1,2,3} | Smooth continuous annual + semi-annual cycle |
| **City dummies** | `city_Baku`, `city_Ganja`, … | Pooled models can share knowledge across cities |

## Data-leak policy (why this code is safe)

Every time-dependent feature is **strictly backward-looking**.
- Lag: `df.groupby("City")[col].shift(k)` — no current value at row `t`.
- Rolling: `series.shift(1).rolling(w).agg()` — window is `t-w .. t-1`, **not** `t-w+1 .. t`.

The notebook executes four explicit unit-style checks at the end to prove this numerically.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

from src.weather import features as fe
from src.weather.features import FeatureConfig, DAILY_TARGET_COLUMNS
from src.utils.config import INTERIM_DIR, PROCESSED_DIR
from src.utils.logging_utils import get_logger
logger = get_logger("nb.03_features")

## 2. Load the cleaned daily frame

In [ ]:
daily = pd.read_csv(INTERIM_DIR / "weather_daily_clean.csv", parse_dates=["date"])
if daily["date"].dt.tz is not None:
    daily["date"] = daily["date"].dt.tz_convert("UTC").dt.tz_localize(None)

print(f"Shape: {daily.shape}, cities: {sorted(daily['City'].unique())}")
print(f"Date range: {daily['date'].min().date()} -> {daily['date'].max().date()}")
print("\nForecast targets (raw -> daily column):")
for k, v in DAILY_TARGET_COLUMNS.items():
    print(f"  {k:20s} -> {v}")

## 3. Walkthrough: build each feature family one at a time

Each step is purely additive — nothing is removed — so you can see exactly
what each helper produces.

### 3a. Wind vector features

In [ ]:
step1 = fe.add_wind_vector_features(daily)
step1[["City", "date", "wind_speed_10m_mean", "wind_direction_10m",
       "wind_u", "wind_v", "wind_dir_sin", "wind_dir_cos"]].head(3)

### 3b. Lag features (demonstrating no leakage)

In [ ]:
step2 = fe.add_lag_features(step1, ["temperature_2m_mean"], lags=[1, 7], group_col="City")
baku = step2[step2["City"]=="Baku"].head(10).reset_index(drop=True)
baku[["date", "temperature_2m_mean",
      "temperature_2m_mean_lag_1", "temperature_2m_mean_lag_7"]]

### 3c. Rolling statistics (shift(1) then rolling — past-only)

In [ ]:
step3 = fe.add_rolling_features(step2, ["temperature_2m_mean"], windows=[7], aggs=["mean", "std"])
baku = step3[step3["City"]=="Baku"].iloc[5:15].reset_index(drop=True)
baku[["date", "temperature_2m_mean",
      "temperature_2m_mean_roll7_mean", "temperature_2m_mean_roll7_std"]]

### 3d. Calendar features

In [ ]:
step4 = fe.add_calendar_features(step3)
step4[["date", "day_of_year", "day_of_week", "month", "quarter", "is_weekend", "year"]].head(5)

### 3e. Fourier features — smooth seasonal encoding

Unlike one-hot `month`, these are continuous everywhere (no sudden jump
between Dec 31 and Jan 1), which linear and neural models handle much better.

In [ ]:
step5 = fe.add_fourier_features(step4, n_harmonics=3)
fourier_cols = [c for c in step5.columns if c.startswith("annual_")]

sub = step5[step5["City"]=="Baku"].set_index("date")[fourier_cols].iloc[:400]
fig, ax = plt.subplots(figsize=(12, 3.5))
for c in fourier_cols:
    ax.plot(sub.index, sub[c], lw=1.0, alpha=0.9, label=c)
ax.set_title("Fourier annual features (first ~13 months)")
ax.legend(ncol=3, fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Full pipeline — `build_weather_features`

This calls every helper in the correct order with a typed `FeatureConfig`.
Reproducible and comparable across experiments.

In [ ]:
cfg = FeatureConfig(
    lags=(1, 2, 3, 7, 14, 30, 365),
    rolling_windows=(3, 7, 14, 30),
    rolling_aggs=("mean", "std", "min", "max"),
    diff_periods=(1, 7),
    fourier_harmonics=3,
    include_weekly_fourier=False,
    include_city_dummies=True,
)
print(cfg)

In [ ]:
feats = fe.build_weather_features(config=cfg, save=True)
print(f"Output: {feats.shape[0]:,} rows x {feats.shape[1]} cols")

## 5. Leakage unit checks

If any of these fail, the whole pipeline is compromised.

In [ ]:
baku_f = feats[feats["City"]=="Baku"].sort_values("date").reset_index(drop=True)

checks = {
    "lag_1 at t equals target at t-1":
        np.isclose(baku_f["temperature_2m_mean_lag_1"].iloc[2],
                   baku_f["temperature_2m_mean"].iloc[1]),
    "lag_7 at t equals target at t-7":
        np.isclose(baku_f["temperature_2m_mean_lag_7"].iloc[10],
                   baku_f["temperature_2m_mean"].iloc[3]),
    "roll7_mean at t equals mean(t-7..t-1)":
        np.isclose(baku_f["temperature_2m_mean_roll7_mean"].iloc[10],
                   baku_f["temperature_2m_mean"].iloc[3:10].mean()),
    "diff_1 at t equals x(t) - x(t-1)":
        np.isclose(baku_f["temperature_2m_mean_diff_1"].iloc[5],
                   baku_f["temperature_2m_mean"].iloc[5] -
                   baku_f["temperature_2m_mean"].iloc[4]),
}

for name, ok in checks.items():
    mark = "✅" if ok else "❌"
    print(f"  {mark} {name}")
assert all(checks.values()), "One or more leakage checks failed!"

## 6. Feature breakdown by family

In [ ]:
meta = {"City", "date"}
cat = {
    "wind_vectors":  [c for c in feats.columns if c in ("wind_u","wind_v","wind_dir_sin","wind_dir_cos")],
    "lags":          [c for c in feats.columns if "_lag_" in c],
    "rolling":       [c for c in feats.columns if "_roll" in c],
    "diffs":         [c for c in feats.columns if "_diff_" in c],
    "calendar":      [c for c in feats.columns if c in ("day_of_year","day_of_week","month","quarter","is_weekend","year")],
    "fourier":       [c for c in feats.columns if c.startswith("annual_") or c.startswith("weekly_")],
    "city_dummies":  [c for c in feats.columns if c.startswith("city_")],
    "outlier_flags": [c for c in feats.columns if c.endswith("_outlier")],
}
cat["base"] = [c for c in feats.columns
               if c not in meta and not any(c in v for v in cat.values())]

pd.DataFrame({"family": list(cat.keys()),
              "n_features": [len(v) for v in cat.values()]})

## 7. Lag-1 autocorrelation sanity check

For temperature, the lag-1 value should correlate extremely highly with the
current value (r > 0.95 is typical). If it doesn't, something is wrong.

In [ ]:
target = "temperature_2m_mean"
valid = feats[feats[f"{target}_lag_1"].notna()]
corr_by_city = (
    valid.groupby("City")
         .apply(lambda g: g[target].corr(g[f"{target}_lag_1"]), include_groups=False)
         .round(4)
)
corr_by_city

Also: lag-365 (same day last year) should still correlate strongly for
temperature — the annual cycle is hugely predictable.

In [ ]:
valid = feats[feats[f"{target}_lag_365"].notna()]
corr_by_city_365 = (
    valid.groupby("City")
         .apply(lambda g: g[target].corr(g[f"{target}_lag_365"]), include_groups=False)
         .round(4)
)
corr_by_city_365

## 8. Train / test time split (preview for Phase 2 Step 3)

In [ ]:
train, test = fe.time_train_test_split(feats, test_start="2025-01-01")
print(f"Train: {train.shape}  ({train['date'].min().date()} .. {train['date'].max().date()})")
print(f"Test : {test.shape}  ({test['date'].min().date()} .. {test['date'].max().date()})")

predictors = fe.feature_columns(feats, exclude_targets=True)
print(f"\nPredictor columns available: {len(predictors)}")
print("First 20:", predictors[:20])

## 9. NaN handling recommendation

The first `max(lags)` rows per city (i.e. the first ~365 days of 2020) will
always have NaN lag-365 features. Three options for Phase 2.3:

1. **Drop** those rows (simplest, loses ~16% of data).
2. **Use fewer long lags** for the earliest years (but keeps the schema ragged).
3. **Train two models** — one with long lags for 2021+ and one without
   for 2020. More complex but uses all data.

We'll revisit in `04_weather_modeling.ipynb` and recommend option 1 by
default — 2020 is a noisy COVID year anyway.

In [ ]:
# Distribution of NaN rows per city
na_rate = (feats.groupby("City").apply(lambda g: g.isna().any(axis=1).mean(), include_groups=False) * 100).round(2)
print(f"Any-NaN rate per city (%):")
print(na_rate.to_string())

# After dropping the first 365 rows per city (the lag-365 warmup period)
trimmed = feats.groupby("City").apply(lambda g: g.iloc[365:], include_groups=False).reset_index(drop=True)
print(f"\nAfter warmup drop: {trimmed.shape[0]:,} rows, "
      f"{trimmed.isna().any(axis=1).mean()*100:.2f}% have any NaN")

---
**Step 2 of Phase 2 ✅ complete.** Next: `04_weather_modeling.ipynb` — SARIMA / Prophet / XGBoost.